<a href="https://colab.research.google.com/github/BlancaHoma14/tecnm-lenguajes-automatas-2/blob/main/laboratorios/sesion28/sesion28_laboratorio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lenguajes y Autómatas II (SCD-1016)
## Sesión 28 · Optimización Local: Plegado de Constantes (Constant Folding)

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Para qué calcular en tiempo de ejecución algo que ya se sabe al compilar?
>
> **Objetivo de la sesión:** Reducir instrucciones TAC aplicando plegado de constantes e identidades algebraicas, sin cambiar el resultado del programa.

**Entregable:** tu función plegar() en artesano_sesion28_c.c, el harness en verde y la bitácora en auditorias/auditoria_sesion28.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)

> ¿Dudas de C? Consulta la [guía de C](https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2/blob/main/docs/guia-c.md) antes de empezar.


## 1. ¿Qué vas a construir?

Recibes el nombre del resultado, el operador y los dos operandos de una instrucción TAC. Debes escribir en dst la versión más simple que conserve el significado: un literal calculado, una identidad resuelta o la instrucción original.

Al final de la sesión tendrás: **Una función en C que pliega constantes, aplica identidades y deja intactas las operaciones que no puede resolver.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 28](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion28.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion28.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion28.h"

void plegar(char *dst, int cap, const char *res, const char *op, const char *a, const char *b);
```

- Solo pliega cuando ambos operandos son enteros completos; nada de prefijos ni texto sobrante.
- x + 0 y x * 1 se reducen a x; x * 0 se reduce a 0.
- Si el divisor es cero, no pliegues: deja la instrucción original y no provoques una excepción.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion28_c.c` junto con `artesano_sesion28_c.c`
> (o `ia_sesion28_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `3 + 5` | `_t0 = 8` |
| 2 | `2 * 3` | `_t0 = 6` |
| 3 | `7 - 10` | `_t0 = -3` |
| 4 | `x + 0` | `_t0 = x` |
| 5 | `x + y (sin plegar)` | `_t0 = x + y` |
| 6 | `10 / 0 (no plegar)` | `_t0 = 10 / 0` |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 28 - Plegado de constantes e identidades ===
Caso                               esperado:              obtenido:              resultado
[ 1] 3 + 5                              esperado: _t0 = 8      obtenido: _t0 = 8      OK
[ 2] 2 * 3                              esperado: _t0 = 6      obtenido: _t0 = 6      OK
[ 3] 7 - 10                             esperado: _t0 = -3     obtenido: _t0 = -3     OK
[ 4] x + 0                              esperado: _t0 = x      obtenido: _t0 = x      OK
[ 5] x + y (sin plegar)                 esperado: _t0 = x + y  obtenido: _t0 = x + y  OK
[ 6] 10 / 0 (no plegar)                 esperado: _t0 = 10 / 0 obtenido: _t0 = 10 / 0 OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion28.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] Dos literales enteros se sustituyen por un solo literal con el resultado calculado.
- [ ] Las identidades x + 0, x * 1 y x * 0 se resuelven sin emitir la operación.
- [ ] Una división entre cero se conserva intacta y no aborta el programa.


## Paso 0 · Entorno


In [1]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


✅ Entorno listo:
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion28_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [2]:
%%writefile api_sesion28.h
/* api_sesion28.h - contrato de la Sesion 28
 * Plegado de constantes e identidades
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION28_H
#define API_SESION28_H

void plegar(char *dst, int cap, const char *res, const char *op, const char *a, const char *b);

#endif /* API_SESION28_H */


Writing api_sesion28.h


In [3]:
%%writefile artesano_sesion28_c.c

#include "api_sesion28.h"
#include <stdio.h>
#include <stdlib.h>
#include <errno.h>
#include <limits.h>
#include <string.h>

static int entero_completo(const char *texto, long long *valor) {
    char *fin;
    long long n;

    if (texto == NULL || texto[0] == '\0') {
        return 0;
    }

    errno = 0;
    n = strtoll(texto, &fin, 10);

    if (errno == ERANGE || fin == texto || *fin != '\0') {
        return 0;
    }

    *valor = n;
    return 1;
}

static int calcular(char op, long long a, long long b,
                    long long *resultado) {
    switch (op) {
        case '+':
            if ((b > 0 && a > LLONG_MAX - b) ||
                (b < 0 && a < LLONG_MIN - b)) {
                return 0;
            }
            *resultado = a + b;
            return 1;

        case '-':
            if ((b < 0 && a > LLONG_MAX + b) ||
                (b > 0 && a < LLONG_MIN + b)) {
                return 0;
            }
            *resultado = a - b;
            return 1;

        case '*':
            if (a > 0) {
                if ((b > 0 && a > LLONG_MAX / b) ||
                    (b < 0 && b < LLONG_MIN / a)) {
                    return 0;
                }
            } else if (a < 0) {
                if ((b > 0 && a < LLONG_MIN / b) ||
                    (b < 0 && a < LLONG_MAX / b)) {
                    return 0;
                }
            }
            *resultado = a * b;
            return 1;

        case '/':
            if (b == 0 || (a == LLONG_MIN && b == -1)) {
                return 0;
            }
            *resultado = a / b;
            return 1;

        default:
            return 0;
    }
}

void plegar(char *dst, int cap, const char *res,
            const char *op, const char *a, const char *b) {
    long long x, y, z;
    int es_a, es_b;
    int n;

    if (dst == NULL || cap <= 0) {
        return;
    }

    dst[0] = '\0';

    if (res == NULL || op == NULL || a == NULL || b == NULL) {
        return;
    }

    es_a = entero_completo(a, &x);
    es_b = entero_completo(b, &y);

    if (strlen(op) == 1 && es_a && es_b &&
        calcular(op[0], x, y, &z)) {
        n = snprintf(dst, (size_t)cap, "%s = %lld", res, z);
    } else if (strlen(op) == 1 && op[0] == '+' &&
               ((es_b && y == 0) || (es_a && x == 0))) {
        n = snprintf(dst, (size_t)cap, "%s = %s",
                     res, es_b && y == 0 ? a : b);
    } else if (strlen(op) == 1 && op[0] == '*' &&
               ((es_b && y == 1) || (es_a && x == 1))) {
        n = snprintf(dst, (size_t)cap, "%s = %s",
                     res, es_b && y == 1 ? a : b);
    } else if (strlen(op) == 1 && op[0] == '*' &&
               ((es_b && y == 0) || (es_a && x == 0))) {
        n = snprintf(dst, (size_t)cap, "%s = 0", res);
    } else {
        n = snprintf(dst, (size_t)cap, "%s = %s %s %s",
                     res, a, op, b);
    }

    if (n < 0 || n >= cap) {
        dst[0] = '\0';
    }
}



Writing artesano_sesion28_c.c


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion28_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


**CONTRATO C-R-E-O**

**CONTEXTO (C):** Estoy desarrollando la Sesión 28 de Lenguajes y Autómatas II (SCD-1016). Necesito implementar una función de optimización local para código intermedio TAC mediante plegado de constantes e identidades algebraicas. El código debe ser compatible con C11 y GCC.

**ROL (R):** Actúa como ingeniero de software especializado en compiladores, optimización de código intermedio, programación en C y validación de entradas.

**ESPECIFICACIÓN (E):** Implementa exactamente esta función:
void plegar(char *dst, int cap, const char *res,
            const char *op, const char *a, const char *b);
Requisitos:
1. Si ambos operandos son enteros completos, calcular las operaciones +, -, * y /.
2. No aceptar números con texto sobrante, como 12abc.
3. Aplicar las identidades x + 0 = x, x * 1 = x y x * 0 = 0, incluyendo el orden inverso cuando corresponda.
4. No plegar una división entre cero.
5. Conservar la instrucción original cuando no sea posible optimizar.
6. Comprobar apuntadores y capacidad del destino.
7. Evitar desbordamientos aritméticos y de conversión.
8. Comprobar el resultado de snprintf().
9. No utilizar memoria dinámica ni bibliotecas externas.
10. No modificar la firma ni agregar main().

**OUTPUT (O):** Entrega el código completo y modular, compatible con:

gcc -std=c11 -Wall -Wextra -Werror

Explica las mejoras respecto a la versión manual, los casos límite considerados y la complejidad temporal y espacial.

In [4]:
%%writefile ia_sesion28_c.c

#include "api_sesion28.h"
#include <stdio.h>
#include <stdlib.h>
#include <errno.h>
#include <limits.h>
#include <string.h>

static int convertir_entero(const char *texto, long long *valor) {
    char *fin;
    long long numero;

    if (texto == NULL || *texto == '\0') {
        return 0;
    }

    errno = 0;
    numero = strtoll(texto, &fin, 10);

    if (errno == ERANGE || fin == texto || *fin != '\0') {
        return 0;
    }

    *valor = numero;
    return 1;
}

static int operar_seguro(char op, long long a,
                         long long b, long long *r) {
    switch (op) {
        case '+':
            if ((b > 0 && a > LLONG_MAX - b) ||
                (b < 0 && a < LLONG_MIN - b)) {
                return 0;
            }
            *r = a + b;
            return 1;

        case '-':
            if ((b < 0 && a > LLONG_MAX + b) ||
                (b > 0 && a < LLONG_MIN + b)) {
                return 0;
            }
            *r = a - b;
            return 1;

        case '*':
            if (a > 0) {
                if ((b > 0 && a > LLONG_MAX / b) ||
                    (b < 0 && b < LLONG_MIN / a)) {
                    return 0;
                }
            } else if (a < 0) {
                if ((b > 0 && a < LLONG_MIN / b) ||
                    (b < 0 && a < LLONG_MAX / b)) {
                    return 0;
                }
            }
            *r = a * b;
            return 1;

        case '/':
            if (b == 0 || (a == LLONG_MIN && b == -1)) {
                return 0;
            }
            *r = a / b;
            return 1;

        default:
            return 0;
    }
}

static int escribir(char *dst, int cap, const char *formato,
                    const char *res, const char *valor) {
    int n = snprintf(dst, (size_t)cap, formato, res, valor);

    if (n < 0 || n >= cap) {
        dst[0] = '\0';
        return 0;
    }

    return 1;
}

void plegar(char *dst, int cap, const char *res,
            const char *op, const char *a, const char *b) {
    long long x = 0, y = 0, z = 0;
    int es_a, es_b;
    int n;
    char resultado[32];

    if (dst == NULL || cap <= 0) {
        return;
    }

    dst[0] = '\0';

    if (res == NULL || op == NULL || a == NULL || b == NULL) {
        return;
    }

    es_a = convertir_entero(a, &x);
    es_b = convertir_entero(b, &y);

    if (op[0] != '\0' && op[1] == '\0' &&
        es_a && es_b && operar_seguro(op[0], x, y, &z)) {
        n = snprintf(resultado, sizeof(resultado), "%lld", z);

        if (n < 0 || (size_t)n >= sizeof(resultado)) {
            return;
        }

        (void)escribir(dst, cap, "%s = %s", res, resultado);
        return;
    }

    if (op[0] == '+' && op[1] == '\0') {
        if (es_b && y == 0) {
            (void)escribir(dst, cap, "%s = %s", res, a);
            return;
        }
        if (es_a && x == 0) {
            (void)escribir(dst, cap, "%s = %s", res, b);
            return;
        }
    }

    if (op[0] == '*' && op[1] == '\0') {
        if (es_b && y == 1) {
            (void)escribir(dst, cap, "%s = %s", res, a);
            return;
        }
        if (es_a && x == 1) {
            (void)escribir(dst, cap, "%s = %s", res, b);
            return;
        }
        if ((es_b && y == 0) || (es_a && x == 0)) {
            (void)snprintf(dst, (size_t)cap, "%s = 0", res);
            if (strlen(dst) >= (size_t)cap - 1) {
                dst[0] = '\0';
            }
            return;
        }
    }

    n = snprintf(dst, (size_t)cap, "%s = %s %s %s",
                 res, a, op, b);

    if (n < 0 || n >= cap) {
        dst[0] = '\0';
    }
}


Writing ia_sesion28_c.c


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [5]:
%%writefile harness_sesion28_c.c
/* harness_sesion28_c.c - auditoria automatica de la Sesion 28
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion28_c.c artesano_sesion28_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion28.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 28 - Plegado de constantes e identidades ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { char b[64] = ""; plegar(b, 64, "_t0", "+", "3", "5"); verificar_str("3 + 5", "_t0 = 8", b); }
    { char b[64] = ""; plegar(b, 64, "_t0", "*", "2", "3"); verificar_str("2 * 3", "_t0 = 6", b); }
    { char b[64] = ""; plegar(b, 64, "_t0", "-", "7", "10"); verificar_str("7 - 10", "_t0 = -3", b); }
    { char b[64] = ""; plegar(b, 64, "_t0", "+", "x", "0"); verificar_str("x + 0", "_t0 = x", b); }
    { char b[64] = ""; plegar(b, 64, "_t0", "+", "x", "y"); verificar_str("x + y (sin plegar)", "_t0 = x + y", b); }
    { char b[64] = ""; plegar(b, 64, "_t0", "/", "10", "0"); verificar_str("10 / 0 (no plegar)", "_t0 = 10 / 0", b); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion28.md" "\n");
    return 0;
}


Writing harness_sesion28_c.c


In [6]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion28_c.c artesano_sesion28_c.c -o test_artesano && ./test_artesano

=== AUDITORIA SESION 28 - Plegado de constantes e identidades ===
Caso                               esperado:              obtenido:              resultado
[ 1] 3 + 5                              esperado: _t0 = 8      obtenido: _t0 = 8      OK
[ 2] 2 * 3                              esperado: _t0 = 6      obtenido: _t0 = 6      OK
[ 3] 7 - 10                             esperado: _t0 = -3     obtenido: _t0 = -3     OK
[ 4] x + 0                              esperado: _t0 = x      obtenido: _t0 = x      OK
[ 5] x + y (sin plegar)                 esperado: _t0 = x + y  obtenido: _t0 = x + y  OK
[ 6] 10 / 0 (no plegar)                 esperado: _t0 = 10 / 0 obtenido: _t0 = 10 / 0 OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion28.md


In [7]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion28_c.c ia_sesion28_c.c -o test_ia && ./test_ia

=== AUDITORIA SESION 28 - Plegado de constantes e identidades ===
Caso                               esperado:              obtenido:              resultado
[ 1] 3 + 5                              esperado: _t0 = 8      obtenido: _t0 = 8      OK
[ 2] 2 * 3                              esperado: _t0 = 6      obtenido: _t0 = 6      OK
[ 3] 7 - 10                             esperado: _t0 = -3     obtenido: _t0 = -3     OK
[ 4] x + 0                              esperado: _t0 = x      obtenido: _t0 = x      OK
[ 5] x + y (sin plegar)                 esperado: _t0 = x + y  obtenido: _t0 = x + y  OK
[ 6] 10 / 0 (no plegar)                 esperado: _t0 = 10 / 0 obtenido: _t0 = 10 / 0 OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion28.md


## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion28.md` a `auditorias/auditoria_sesion28.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
